In [1]:
import polars as pr
import numpy as np
import pandas as pd
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
BASE_PATH = os.environ.get('BASE_PATH')
df = pr.read_parquet(
    f"{BASE_PATH}/data/processed/df_merged.parquet",
    columns=["item_store_id", "day_number", "sales"]
)

print(df.shape)
print(df.dtypes)

(59181090, 3)
[String, Int16, Int16]


In [14]:
df.schema

Schema([('item_store_id', String), ('day_number', Int16), ('sales', Int16)])

In [15]:
df.null_count()

item_store_id,day_number,sales
u32,u32,u32
0,0,0


In [16]:
df['sales'].value_counts()

sales,count
i16,u32
145,50
180,22
105,202
51,2304
248,7
…,…
8,305787
132,68
80,544


In [18]:
df = df.sort(["item_store_id", "day_number"])

df = df.with_columns([
    pr.col("sales").shift(7).over("item_store_id").alias("lag_7"),
    pr.col("sales").shift(28).over("item_store_id").alias("lag_28"),
])

df.head(20)

item_store_id,day_number,sales,lag_7,lag_28
str,i16,i16,i16,i16
"""FOODS_1_001_CA_1_evaluation""",1,3,null,null
"""FOODS_1_001_CA_1_evaluation""",2,0,null,null
"""FOODS_1_001_CA_1_evaluation""",3,0,null,null
"""FOODS_1_001_CA_1_evaluation""",4,1,null,null
"""FOODS_1_001_CA_1_evaluation""",5,4,null,null
…,…,…,…,…
"""FOODS_1_001_CA_1_evaluation""",16,0,0,null
"""FOODS_1_001_CA_1_evaluation""",17,2,0,null
"""FOODS_1_001_CA_1_evaluation""",18,1,0,null


In [ ]:
df.write_parquet(
    f"{BASE_PATH}/data/processed/lag_features.parquet"
)
